# CHARTEVENTS - Bronze Layer

This notebook ingests the MIMIC-IV `chartevents` source table and stores it as a Bronze Delta table.

A small source check is completed before ingestion to confirm the table structure and key identifiers.

In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
BRONZE_SCHEMA = "bronze"

SOURCE_PATH = "s3://mimic-iii-demo-bucket/mimic-iv-dataset-demo-2.2-folder/icu/chartevents.csv"

chartevents = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(SOURCE_PATH)
)

print(f"Rows: {chartevents.count():,}")
print(f"Columns: {len(chartevents.columns)}")

chartevents.printSchema()

### Source Check

`CHARTEVENTS` contains patient observations recorded during ICU stays.

The main patient, stay, item and caregiver identifiers are checked for missing values before the source data is stored in Bronze.

In [0]:
# Check core fields
key_columns = [
    "subject_id",
    "hadm_id",
    "stay_id",
    "caregiver_id",
    "itemid",
    "charttime"
]

for column in key_columns:
    nulls = chartevents.filter(F.col(column).isNull()).count()
    print(f"{column}: {nulls:,} nulls")

### Source Check

The main patient, admission and ICU stay identifiers are complete.

`caregiver_id` is missing for some chart events, but this is kept unchanged in Bronze because the raw source structure should be preserved. Any interpretation or validation of missing caregiver links will be handled in Silver.

In [0]:
# Add ingestion metadata
chartevents_bronze = (
    chartevents
    .withColumn("_source_file", F.lit("chartevents.csv"))
    .withColumn("_source_module", F.lit("icu"))
    .withColumn("_ingestion_timestamp", F.current_timestamp())
)

# Save as Delta table
(
    chartevents_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.chartevents_raw")
)

print("Created: workspace.bronze.chartevents_raw")

In [0]:
source_count = chartevents.count()
bronze_count = spark.table("workspace.bronze.chartevents_raw").count()

print(f"Source: {source_count:,}")
print(f"Bronze: {bronze_count:,}")
print(f"Match: {source_count == bronze_count}")